<!-- notebook-header -->
# Pre-Calculo e Funcoes para Machine Learning

**Modulo:** 00 - Matematica  
**Tipo:** Aula com exercicios guiados e solucoes executaveis  
**Descricao:** Revisao de funcoes, algebra basica, exponenciais, logaritmos e ativacoes usadas em ML.


# Pre-Calculo e Funcoes para Machine Learning

**Construindo a linguagem matemática que sustenta todos os algoritmos de ML**

---

## Índice

1. [Por que Matemática para ML?](#1-por-que-matematica)
2. [Revisão de Álgebra Básica](#2-algebra-basica)
3. [Funções: Conceitos Fundamentais](#3-funcoes-fundamentais)
4. [Funções Lineares → Regressão Linear](#4-funcoes-lineares)
5. [Funções Quadráticas e Polinomiais → Features Polinomiais](#5-funcoes-polinomiais)
6. [Funções Exponenciais → Softmax e Decaimento](#6-funcoes-exponenciais)
7. [Funções Logarítmicas → Cross-Entropy e Entropia](#7-funcoes-logaritmicas)
8. [Sigmoid e Funções de Ativação → Redes Neurais](#8-sigmoid-ativacoes)
9. [Somatórios e Produtórios → Funções de Custo e MLE](#9-somatorios-produtorios)
10. [Exercícios Práticos](#10-exercicios)

---

**Pré-requisitos:** Matemática básica (ensino médio)
**Tempo estimado:** 6–8 horas
**Próximo notebook:** `0_2_algebra_linear_vetores.ipynb`

### O que concluir: Funções de Perda

- **Cross-entropy penaliza confiança errada:** Se prever 0.99 para classe errada, perda é gigante
- **Log(p) torna pequenas diferenças grandes:** Diferença entre 0.9 e 0.99 é mais relevante que entre 0.1 e 0.2
- **Combina com softmax:** Softmax + cross-entropy é a dupla padrão para classificação

---
## 1. Por que Matematica para ML? <a id='1-por-que-matematica'></a>

Machine Learning e, fundamentalmente, **matematica aplicada**. Cada algoritmo que voce
vai aprender ao longo desta serie tem uma fundacao matematica precisa.

Entender essa fundacao permite que voce:

| Sem Matematica | Com Matematica |
|----------------|----------------|
| Usa algoritmos como "caixas pretas" | Entende *por que* um algoritmo funciona |
| Depende de tentativa e erro para ajustes | Sabe exatamente qual parametro ajustar |
| Nao consegue diagnosticar problemas | Identifica overfitting, gradientes explodindo, etc. |
| Dificuldade em ler artigos cientificos | Le equacoes e entende novas tecnicas |

### O Mapa da Matematica no ML

```
Funcoes          ->  Representar relacoes entre variaveis (features -> target)
Derivadas        ->  Gradient Descent (como treinar modelos)
Algebra Linear   ->  Operacoes com dados (matrizes de features, embeddings)
Probabilidade    ->  Incerteza, distribuicoes, Naive Bayes, VAEs
Otimizacao       ->  Encontrar os melhores parametros do modelo
```

Neste primeiro notebook, focamos nas **funcoes** -- o bloco mais fundamental.

### Como Pensar Sobre Funcoes em ML

Imagine que voce esta construindo um modelo para prever o preco de casas. Voce tem dados como area, numero de quartos, localizacao. O que o modelo faz e essencialmente aprender uma **funcao** que mapeia essas caracteristicas para um preco:

```
f(area, quartos, localizacao) = preco_estimado
```

Toda a matematica que vamos estudar serve para responder tres perguntas:
1. **Qual funcao escolher?** (Este notebook -- tipos de funcoes)
2. **Como medir se a funcao e boa?** (Funcoes de custo -- secao 9)
3. **Como melhorar a funcao?** (Derivadas e otimizacao -- notebooks 0.4 e 0.8)

In [ ]:
# ── Configuração do Ambiente ──────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyArrowPatch

# Configurações globais de visualização
plt.rcParams.update({
    'figure.figsize': (10, 5),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})
np.random.seed(42)
print("Ambiente configurado com sucesso!")


### Erro: Logaritmo de Valores Inválidos

**Erro 1: Aplicar log(0) ou em negativos**
```python
# ERRADO
loss = -np.log(probs)  # probs contém 0? → -inf
loss = np.log(-values)  # → nan

# CORRETO: adiciona pequeno epsilon
epsilon = 1e-15
loss = -np.log(np.clip(probs, epsilon, 1.0))
loss = np.log(np.abs(values) + epsilon)
```

**Por que:** Log(0) = -∞, log(negativo) = NaN. Sempre proteja com clip() ou valores garantidamente positivos.

---
## 2. Revisão de Álgebra Básica <a id='2-algebra-basica'></a>

A álgebra é a base de tudo. Em ML, você vai encontrar expressões como:

$$\hat{y} = w_0 + w_1 x_1 + w_2 x_2 + \cdots + w_n x_n$$

Vamos revisar os conceitos essenciais que aparecem constantemente.

### 2.1 Variáveis, Expressões e Equações

- **Variável:** símbolo que representa um valor desconhecido (`x`, `w`, `θ`)
- **Expressão:** combinação de variáveis e operações (`2x + 3`)
- **Equação:** igualdade entre duas expressões (`y = 2x + 3`)

### 2.2 Operações Fundamentais

| Operação | Notação | Exemplo em ML |
|----------|---------|---------------|
| Potência | $x^n$ | Regularização L2: $w^2$ |
| Raiz | $\sqrt{x}$ | Norma Euclidiana: $\sqrt{\sum x_i^2}$ |
| Fração | $a/b$ | Taxa de aprendizado: $\eta = 1/t$ |
| Produto | $a \cdot b$ ou $ab$ | Produto escalar: $w \cdot x$ |

### 2.3 Propriedades Essenciais

```
Comutativa:   a + b = b + a
Distributiva: a(b + c) = ab + ac  ← Muito usada em expansões de custo!
Potências:    x^a · x^b = x^(a+b)
              (x^a)^b = x^(ab)
Logaritmos:   log(ab) = log(a) + log(b)  ← Fundamental para MLE!
              log(a^b) = b·log(a)
```

### O que concluir: Propriedades Algébricas

- **Propriedades são imutáveis:** Comutatividade, associatividade e distributividade funcionam SEMPRE
- **Aparecem em backpropagation:** Gradientes envolvem essas propriedades repetidamente
- **Simplificação = velocidade:** Usar essas leis permite simplificar cálculos antes de implementar

In [ ]:
# Verificando propriedades algébricas com NumPy
# Esses truques aparecem em TODA implementação de ML

# 1. Propriedade distributiva → usada em expansão de gradientes
a, b, c = 3.0, 4.0, 5.0
print(f"Distributiva: a(b+c) = {a*(b+c):.1f} | ab+ac = {a*b + a*c:.1f}")

# 2. Propriedades de potência → regularização L2
w = np.array([0.5, -1.2, 0.8, -0.3])
l2_norm = np.sqrt(np.sum(w**2))
l2_norm_alt = np.linalg.norm(w)  # mesmo resultado
print(f"\nNorma L2 de w: {l2_norm:.4f} (via sum) | {l2_norm_alt:.4f} (via linalg)")

# 3. Logaritmos → transformar produtos em somas (MLE!)
probs = np.array([0.8, 0.7, 0.9, 0.6])  # probabilidades hipotéticas
log_likelihood_prod = np.prod(probs)  # produto direto (problema: underflow numérico!)
log_likelihood_sum  = np.sum(np.log(probs))  # soma de logs (estável numericamente)
print(f"\nLog-likelihood:")
print(f"  Produto direto: {log_likelihood_prod:.6f}  (pode virar 0 com muitos dados!)")
print(f"  Soma dos logs:  {log_likelihood_sum:.6f}  (estável numericamente)")

# 4. Completar o quadrado → aparece na solução de regressão linear
# (x + h)^2 = x^2 + 2hx + h^2
x = np.linspace(-3, 3, 100)
h = 1.0
expanded = x**2 + 2*h*x + h**2
factored = (x + h)**2
print(f"\nCompletar o quadrado: max|erro| = {np.max(np.abs(expanded - factored)):.2e}")

### Por que em ML: Álgebra Importa

1. **Multiplicação de matrizes:** Toda transformação linear é multiplicação; precisa ser rápida
2. **Comutatividade não existe:** A·B ≠ B·A; ordem das operações é crítica em redes neurais
3. **Eigenvectors em PCA:** Decomposição usa álgebra para encontrar direções de variância
4. **Simplicidade teórica:** Se entende álgebra básica, derivadas e otimização ficam fáceis

### O que concluir: Conceitos de Funções

- **Domínio e contradomínio importam:** Uma função só é válida se respeita seus domínios
- **Injetividade permite inversa:** Se f é injetiva (1-to-1), existe f⁻¹; crítico para autoencoders
- **Composição é generalizar:** f(g(h(x))) é exatamente como redes neurais profundas funcionam

---
## 3. Funções: Conceitos Fundamentais <a id='3-funcoes-fundamentais'></a>

Uma **função** é uma regra que associa cada elemento de um conjunto (domínio) a
exatamente um elemento de outro conjunto (contradomínio).

$$f: X \rightarrow Y \quad \text{lê-se: "f leva X em Y"}$$

Em ML, você lida com funções constantemente:
- `modelo(x)` → previsão
- `perda(ŷ, y)` → erro
- `ativação(z)` → saída de neurônio

### Conceitos-chave

| Conceito | Definição | Exemplo em ML |
|----------|-----------|---------------|
| **Domínio** | Conjunto de entradas válidas | Features de entrada |
| **Imagem** | Conjunto de saídas possíveis | Previsões do modelo |
| **Injetora** | Cada saída tem no máx. 1 entrada | Funções invertíveis |
| **Sobrejetora** | Toda saída é mapeada | Garante cobertura |
| **Bijetora** | Injetora + Sobrejetora | Invertível (usada em normalizing flows) |
| **Crescente** | $x_1 < x_2 \Rightarrow f(x_1) < f(x_2)$ | ReLU (para x>0) |
| **Decrescente** | $x_1 < x_2 \Rightarrow f(x_1) > f(x_2)$ | Perda vs. acurácia |
| **Contínua** | Sem "saltos" | Necessária para derivação |

### Conexao com Algoritmos de Aprendizado

- **Descontinuidade:** ReLU é descontínua em x=0; isso deixa algumas features "mortas"
- **Invertibilidade:** Autoencoders precisam de funções aproximadamente invertíveis
- **Monotonia:** Funções monôtonas têm inversa; Sigmoid é quase-monôtona, permite decodificação

In [ ]:
# Visualizando propriedades de funções
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

x = np.linspace(-3, 3, 300)

# Função contínua vs. descontinua
axes[0].plot(x, np.sin(x), 'b-', lw=2, label='sin(x) -- contínua')
x_disc = x.copy()
y_disc = np.where(x_disc >= 0, 1.0, -1.0)
axes[0].plot(x_disc, y_disc, 'r--', lw=2, label='sign(x) -- descontinua')
axes[0].axvline(0, color='gray', lw=0.8, ls=':')
axes[0].set_title('Contínua vs. Descontinua')
axes[0].legend()
axes[0].set_xlabel('x')

# Crescente, decrescente, não-monótona
axes[1].plot(x, x**3, 'g-', lw=2, label='x³ -- crescente')
axes[1].plot(x, -x**2, 'r-', lw=2, label='-x² -- decrescente (x>0)')
axes[1].plot(x, np.sin(x), 'b-', lw=2, label='sin(x) -- não-monótona')
axes[1].set_title('Monotonicidade')
axes[1].legend(fontsize=9)
axes[1].set_xlabel('x')

# Domínio restrito (relevante para log, sqrt)
x_pos = np.linspace(0.01, 4, 300)
axes[2].plot(x_pos, np.sqrt(x_pos), 'm-', lw=2, label='√x  (domínio: x≥0)')
axes[2].plot(x_pos, np.log(x_pos),  'c-', lw=2, label='ln(x) (domínio: x>0)')
axes[2].axhline(0, color='gray', lw=0.8)
axes[2].set_title('Domínio Restrito')
axes[2].legend()
axes[2].set_xlabel('x')
axes[2].set_ylim(-3, 3)

plt.suptitle('Propriedades de Funções', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print("Observação: ln(x) vai a -∞ quando x→0⁺ -- isso explica instabilidade numérica!")

### O que concluir: Conceitos de Funções

- **Domínio e contradomínio importam:** Uma função só é válida se respeita seus domínios
- **Injetividade permite inversa:** Se f é injetiva (1-to-1), existe f⁻¹; crítico para autoencoders
- **Composição é generalizar:** f(g(h(x))) é exatamente como redes neurais profundas funcionam

### O que concluir

- **Continuidade importa:** Funções contínuas permitem usar cálculo e derivadas. Descontinuas não.
- **Monotonicidade ajuda:** Funções monotônicas (crescentes ou decrescentes) são mais previsíveis.
- **Domínio restrito é comum:** sqrt(x) e log(x) têm domínios limitados -- cuidado em ML!
- **Composição cria complexidade:** A maioria das redes neurais são composições de funções simples.


---
## 4. Funcoes Lineares -> Regressao Linear <a id='4-funcoes-lineares'></a>

### Intuicao: A Reta Mais Simples do Mundo

Antes de qualquer formula, pense assim: se voce ganha R$50 por hora de trabalho e tem um custo fixo de R$200 de transporte por mes, seu rendimento liquido e:

```
rendimento = 50 * horas - 200
```

Isso e uma funcao linear. O "50" diz o quanto cada hora a mais impacta o resultado (inclinacao), e o "-200" e o ponto de partida (intercepto). Em ML, a logica e identica: queremos descobrir **quanto cada feature impacta a previsao**.

### Definicao Formal

$$f(x) = mx + b$$

onde:
- $m$ = **inclinacao** (slope) -- quanto $y$ muda para cada unidade de $x$
- $b$ = **intercepto** -- valor de $y$ quando $x = 0$

### Por que Funcoes Lineares sao Centrais em ML?

1. **Regressao Linear:** $\hat{y} = w_1 x_1 + w_2 x_2 + \cdots + w_n x_n + b$
2. **Camada Linear em Redes Neurais:** $z = Wx + b$
3. **SVM:** Hiperplano separador $w \cdot x + b = 0$
4. **Regressao Logistica:** A parte linear antes da sigmoid

Mesmo modelos "nao-lineares" como redes neurais profundas sao composicoes de transformacoes lineares intercaladas com funcoes de ativacao nao-lineares. Entender o caso linear e entender o bloco basico de quase tudo.

### Interpretacao da Inclinacao

| Inclinacao $m$ | Comportamento | Exemplo ML |
|----------------|---------------|------------|
| $m > 0$ | Feature positivamente correlacionada com target | Area -> Preco sobe |
| $m < 0$ | Feature negativamente correlacionada | Distancia -> Preco desce |
| $m = 0$ | Feature irrelevante | Feature sem poder preditivo |
| $|m|$ grande | Feature muito influente | Feature importante |

---
## 4. Funções Lineares → Regressão Linear <a id='4-funcoes-lineares'></a>

A função linear é o bloco fundamental da maioria dos algoritmos de ML:

$$f(x) = mx + b$$

onde:
- $m$ = **inclinação** (slope) → quanto $y$ muda para cada unidade de $x$
- $b$ = **intercepto** → valor de $y$ quando $x = 0$

### Por que Funções Lineares são Centrais em ML?

1. **Regressão Linear:** $\hat{y} = w_1 x_1 + w_2 x_2 + \cdots + w_n x_n + b$
2. **Camada Linear em Redes Neurais:** $z = Wx + b$
3. **SVM:** Hiperplano separador $w \cdot x + b = 0$
4. **Regressão Logística:** A parte linear antes da sigmoid

### Interpretação da Inclinação

| Inclinação $m$ | Comportamento | Exemplo ML |
|----------------|---------------|------------|
| $m > 0$ | Feature positivamente correlacionada com target | Área ↑ → Preço ↑ |
| $m < 0$ | Feature negativamente correlacionada | Distância ↑ → Preço ↓ |
| $m = 0$ | Feature irrelevante | Feature sem poder preditivo |
| $|m|$ grande | Feature muito influente | Feature importante |

### O que observar nos graficos acima

**Grafico 1 -- Efeito da inclinacao:** Repare como m=0 gera uma reta horizontal (feature irrelevante) e como m negativo inverte a direcao. Em um modelo real, cada feature tem seu proprio "m" (peso), e o treinamento consiste em encontrar os melhores valores para esses pesos.

**Grafico 2 -- Efeito do intercepto:** Todas as retas tem a mesma inclinacao (m=1), mas partem de pontos diferentes. O intercepto e o "bias" do modelo -- sem ele, todas as retas passariam pela origem, limitando a capacidade do modelo.

### Conexao com Machine Learning

A regressão linear é o algoritmo mais simples de ML, mas ensina os princípios fundamentais:
- **Feature scaling:** Se m é muito grande, pequenas mudanças em x causam grandes mudanças em ŷ
- **Bias-Variance:** Linhas muito retas (bias alto) vs muito onduladas (variância alta)
- **Regularização:** Penalizar valores grandes de |m| evita overfitting


### Por que em ML: Regressão Linear

1. **Interpretabilidade:** Coeficientes têm significado direto (aumento de 1 unidade em x causa aumento de m em y)
2. **Baseline poderoso:** Regressão linear é o primeiro modelo a tentar; se não funciona, o problema é não-linear
3. **Closed-form solution:** Existe fórmula analítica (normal equations), não precisa de iterações
4. **Fundação para tudo:** Todos os algoritmos mais complexos (SVM, redes neurais) começam como combinações lineares

In [ ]:
# Visualizando funções lineares com diferentes inclinações
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

x = np.linspace(-3, 3, 100)

# Plot 1: Efeito da inclinação (m)
slopes = [-2, -0.5, 0, 0.5, 1, 2]
colors = plt.cm.RdYlGn(np.linspace(0.1, 0.9, len(slopes)))
for m, c in zip(slopes, colors):
    axes[0].plot(x, m*x + 0, color=c, lw=2, label=f'm={m}')
axes[0].axhline(0, color='k', lw=0.5); axes[0].axvline(0, color='k', lw=0.5)
axes[0].set_title('Efeito da Inclinação (b=0 fixo)')
axes[0].set_xlabel('x  (feature)'); axes[0].set_ylabel('f(x)  (previsão)')
axes[0].legend(loc='upper left', fontsize=9)

# Plot 2: Efeito do intercepto (b)
intercepts = [-2, -1, 0, 1, 2]
colors2 = plt.cm.Blues(np.linspace(0.3, 0.9, len(intercepts)))
for b, c in zip(intercepts, colors2):
    axes[1].plot(x, 1*x + b, color=c, lw=2, label=f'b={b}')
axes[1].axhline(0, color='k', lw=0.5); axes[1].axvline(0, color='k', lw=0.5)
axes[1].set_title('Efeito do Intercepto (m=1 fixo)')
axes[1].set_xlabel('x  (feature)'); axes[1].set_ylabel('f(x)  (previsão)')
axes[1].legend(loc='upper left', fontsize=9)

plt.suptitle('Funções Lineares: f(x) = mx + b', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 5. Funcoes Quadraticas e Polinomiais -> Features Polinomiais <a id='5-funcoes-polinomiais'></a>

### Intuicao: Quando a Reta Nao Basta

Imagine que voce esta modelando o efeito da temperatura na produtividade de uma fabrica. Ate uns 25 graus, mais calor nao atrapalha. Mas acima de 35 graus, a produtividade despenca. E abaixo de 10, tambem cai. Uma reta nao consegue capturar esse comportamento de "sobe e depois desce" -- voce precisa de uma **parabola** (funcao quadratica).

De forma geral: quando a relacao entre X e Y nao segue uma linha reta, funcoes polinomiais permitem capturar curvas, pontos de inflexao e comportamentos mais complexos.

### Definicao Formal

Uma funcao polinomial de grau $n$ tem a forma:

$$f(x) = a_n x^n + a_{n-1} x^{n-1} + \cdots + a_1 x + a_0$$

### Graus Comuns em ML

| Grau | Nome | Formato | Usos em ML |
|------|------|---------|------------|
| 1 | Linear | Reta | Regressao linear, camada densa |
| 2 | Quadratica | Parabola | Features polinomiais, MSE (funcao de custo!) |
| 3 | Cubica | Curva com inflexao | Aproximacoes locais |
| n | Polinomio | Flexivel | Kernel polinomial no SVM |

### A Funcao Quadratica: $f(x) = ax^2 + bx + c$

O coeficiente $a$ controla a "abertura" da parabola:
- $a > 0$: abre para **cima** (tem minimo) -- forma do MSE!
- $a < 0$: abre para **baixo** (tem maximo)

> **Conexao com ML:** A funcao de custo MSE e uma parabola convexa em relacao
> aos pesos. Isso garante que existe um **unico minimo global**, tornando a
> otimizacao muito mais simples. Funcoes de custo nao-convexas (como em redes neurais
> profundas) tem multiplos minimos locais, o que torna o treinamento mais desafiador.

### O que observar: Inclinação e Interpretação

- **m negativo = relação inversa:** Quando m < 0, aumento em x gera diminuição em y
- **y-intercept (b) = bias:** O valor de y quando x=0 é exatamente o viés do modelo
- **Linearidade é forte:** Uma reta pode explicar ~70% da variância em muitos fenômenos reais

In [ ]:
# Regressão Linear: aprendendo m e b a partir dos dados (implementação manual)
np.random.seed(42)
n_samples = 80

# Gerando dados sintéticos: preço de casas (simplificado)
area = np.random.uniform(50, 200, n_samples)  # m²
price = 3.5 * area + 50 + np.random.normal(0, 25, n_samples)  # preço em R$ mil

# Treinando o modelo (implementação manual usando mínimos quadrados)
# Formula: m = (n*sum(xy) - sum(x)*sum(y)) / (n*sum(x²) - (sum(x))²)
X_area = area
n = len(X_area)
m_learned = (n * np.sum(X_area * price) - np.sum(X_area) * np.sum(price)) / \
            (n * np.sum(X_area**2) - np.sum(X_area)**2)
b_learned = (np.sum(price) - m_learned * np.sum(X_area)) / n

# Calcular R²
y_pred = m_learned * X_area + b_learned
ss_res = np.sum((price - y_pred)**2)
ss_tot = np.sum((price - np.mean(price))**2)
r_squared = 1 - (ss_res / ss_tot)

print(f"Parâmetros aprendidos:")
print(f"  Inclinação (m): {m_learned:.2f}  → cada m² adicional vale ~R${m_learned:.0f}k")
print(f"  Intercepto (b): {b_learned:.2f}  → preço base sem área seria R${b_learned:.0f}k")
print(f"  R²: {r_squared:.4f}")

# Visualização
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Dados + reta ajustada
x_line = np.linspace(40, 210, 100)
y_line = m_learned * x_line + b_learned
axes[0].scatter(area, price, alpha=0.6, color='steelblue', label='Dados')
axes[0].plot(x_line, y_line, 'r-', lw=2.5,
             label=f'f(x) = {m_learned:.2f}x + {b_learned:.1f}')
axes[0].set_xlabel('Área (m²)'); axes[0].set_ylabel('Preço (R$ mil)')
axes[0].set_title('Regressão Linear: ajuste da reta')
axes[0].legend()

# Resíduos (erros)
residuals = price - y_pred
axes[1].scatter(y_pred, residuals, alpha=0.6, color='coral')
axes[1].axhline(0, color='k', lw=1.5, ls='--')
axes[1].set_xlabel('Previsão (ŷ)'); axes[1].set_ylabel('Resíduo (y - ŷ)')
axes[1].set_title('Análise de Resíduos')
axes[1].text(0.05, 0.95, f'MSE: {np.mean(residuals**2):.1f}',
             transform=axes[1].transAxes, va='top')

plt.tight_layout()
plt.show()


### Por que em ML: Polinômios

1. **Aproximação universal:** Qualquer função contínua pode ser aproximada por polinômios (Stone-Weierstrass)
2. **Feature engineering:** Criar features polinomiais permite modelos simples captar não-linearidades
3. **Kernel methods:** SVM usa polinômios implicitamente via kernel trick (eficiente)
4. **Trade-off interpretabilidade:** Polinômio de grau 2 é interpretável; grau 10+ é caixa preta

### O que observar nos graficos acima

**Grafico 1 -- Quadraticas:** Coeficientes positivos (a > 0) criam parabolas que abrem para cima -- tem um ponto de minimo. E exatamente isso que queremos numa funcao de custo: um ponto otimo que o gradient descent pode encontrar. Coeficientes negativos criam maximos.

**Grafico 2 -- MSE como parabola:** Este e o insight central. A funcao de custo MSE, vista como funcao de um unico peso, e uma parabola perfeita. O ponto vermelho e o minimo -- exatamente onde o gradient descent quer chegar. A convexidade garante que nao existem "armadilhas" (minimos locais).

**Grafico 3 -- Aproximacao polinomial:** Observe como polinomios de grau baixo (1, 2) nao conseguem capturar a forma de sin(x), enquanto grau 5 se aproxima bem. Mas cuidado: graus muito altos memorizam ruido (overfitting) -- veremos isso na proxima celula.

---
## 5. Funções Quadráticas e Polinomiais → Features Polinomiais <a id='5-funcoes-polinomiais'></a>

Uma função polinomial de grau $n$ tem a forma:

$$f(x) = a_n x^n + a_{n-1} x^{n-1} + \cdots + a_1 x + a_0$$

### Graus Comuns em ML

| Grau | Nome | Formato | Usos em ML |
|------|------|---------|------------|
| 1 | Linear | Reta | Regressão linear, camada densa |
| 2 | Quadrática | Parábola | Features polinomiais, MSE (função de custo!) |
| 3 | Cúbica | Curva com inflexão | Aproximações locais |
| n | Polinômio | Flexível | Kernel polinomial no SVM |

### A Função Quadrática: $f(x) = ax^2 + bx + c$

O coeficiente $a$ controla a "abertura" da parábola:
- $a > 0$: abre para **cima** (tem mínimo) -- forma do MSE!
- $a < 0$: abre para **baixo** (tem máximo)

> **Conexão com ML:** A função de custo MSE é uma parábola convexa em relação
> aos pesos. Isso garante que existe um **único mínimo global**, tornando a
> otimização muito mais simples.

### O que concluir: Quando Usar Polinômios

- **Trade-off: bias vs variância:** Polinômio de grau alto encaixa tudo mas overfita
- **Regularização é essencial:** L1/L2 penalty previne overfitting em features polinomiais
- **Maldição da dimensionalidade:** Muitas features polinomiais = exponential mais dados necessários

---
## 6. Funcoes Exponenciais -> Softmax e Decaimento <a id='6-funcoes-exponenciais'></a>

### Intuicao: Crescimento que Explode

Pense em juros compostos: se voce investe R$100 com 10% ao mes, depois de 1 mes tem R$110, depois de 2 meses R$121, depois de 3 R$133... O valor nao cresce de forma constante (como uma reta), mas **acelera** -- cada mes o acrescimo e maior que o anterior. Isso e crescimento exponencial.

Em ML, a exponencial aparece em dois contextos opostos:
- **Crescimento:** Softmax usa $e^x$ para amplificar diferencas entre scores (logits maiores viram probabilidades proporcionalmente ainda maiores)
- **Decaimento:** Learning rate scheduling usa $e^{-\lambda t}$ para reduzir a taxa de aprendizado ao longo do treino (passos grandes no inicio, refinamento no final)

### Definicao Formal

$$f(x) = e^x \quad \text{onde } e \approx 2.71828... \text{ (numero de Euler)}$$

### Propriedades Cruciais para ML

| Propriedade | Formula | Por que importa |
|-------------|---------|-----------------|
| Sempre positiva | $e^x > 0 \; \forall x$ | Gera probabilidades validas |
| Monotonica crescente | $e^{x_1} < e^{x_2}$ se $x_1 < x_2$ | Preserva ordenacao |
| Inversa do log | $e^{\ln x} = x$ | Conecta com cross-entropy |
| Crescimento rapido | $e^{10} \approx 22026$ | Causa overflow numerico! |

### A Constante e

$$e = \lim_{n \to \infty} \left(1 + \frac{1}{n}\right)^n \approx 2.71828$$

Em ML, $e$ aparece porque:
1. Suas derivadas tem forma elegante: $(e^x)' = e^x$ -- simplifica backpropagation
2. E a base natural para crescimento/decaimento continuo
3. Aparece na distribuicao Normal: $e^{-x^2/2}$

### Aplicação: Regressão Polinomial

Quando os dados **não** têm relação linear, podemos criar novas features polinomiais:

$$x \rightarrow [x, x^2, x^3, \ldots, x^d]$$

Isso transforma um problema não-linear em um problema linear sobre as novas features!

### Conexao com Currículo

- **0_1_pre_calculo_funcoes_ml.ipynb:** Raízes de polinômios = pontos críticos em otimização
- **0_4_calculo_derivadas.ipynb:** Segunda derivada de quadrática é constante; Hessian é matriz de segundas derivadas
- **3_4_svm_kernel.ipynb:** Kernel trick transforma dados para espaço polinomial implicitamente

### Conexao com Notebooks Posteriores

- **0_4_calculo_derivadas.ipynb:** Derivar polinômios é essencial para otimização
- **0_3_algebra_linear_matrizes.ipynb:** Regressão polinomial usa matriz de features polinomiais
- **4_1_fundamentos_redes_neurais.ipynb:** Camadas densas criam combinações polinomiais dos inputs

### Aplicacao 1: Funcao Softmax

A **Softmax** converte um vetor de scores (logits) em probabilidades:

$$\text{Softmax}(z_i) = \frac{e^{z_i}}{\sum_{j} e^{z_j}}$$

E usada na ultima camada de redes neurais para **classificacao multiclasse**.

#### Por que usar exponencial e nao simplesmente dividir pelo total?

Se fizessemos apenas $z_i / \sum z_j$, valores negativos gerariam "probabilidades" negativas -- o que nao faz sentido. A exponencial resolve isso porque $e^x > 0$ sempre. Alem disso, ela **amplifica diferencas**: se um logit e um pouco maior que os outros, a exponencial o torna proporcionalmente muito maior, criando previsoes mais "confiantes".

#### Problema numerico e a solucao

Se $z_i$ e muito grande (ex: 1000), $e^{1000}$ e um numero com 434 digitos -- o computador nao consegue representar e retorna `inf` (overflow).

**Solucao (Log-Sum-Exp trick):** subtrair $\max(z)$ de todos os logits antes de calcular. Matematicamente, o resultado e identico (as fracoes se cancelam), mas numericamente fica estavel porque o maior expoente vira $e^0 = 1$.

---
## 6. Funções Exponenciais → Softmax e Decaimento <a id='6-funcoes-exponenciais'></a>

A função exponencial é definida como:

$$f(x) = e^x \quad \text{onde } e \approx 2.71828... \text{ (número de Euler)}$$

### Propriedades Cruciais para ML

| Propriedade | Fórmula | Por que importa |
|-------------|---------|-----------------|
| Sempre positiva | $e^x > 0 \; \forall x$ | Gera probabilidades válidas |
| Monotônica crescente | $e^{x_1} < e^{x_2}$ se $x_1 < x_2$ | Preserva ordenação |
| Inversa do log | $e^{\ln x} = x$ | Conecta com cross-entropy |
| Crescimento rápido | $e^{10} \approx 22026$ | Causa overflow numérico! |

### A Constante e

$$e = \lim_{n \to \infty} \left(1 + \frac{1}{n}\right)^n \approx 2.71828$$

Em ML, $e$ aparece porque:
1. Suas derivadas têm forma elegante: $(e^x)' = e^x$
2. É a base natural para crescimento/decaimento contínuo
3. Aparece na distribuição Normal: $e^{-x^2/2}$

In [ ]:
# Função exponencial e suas variantes
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

x = np.linspace(-4, 4, 300)
x_pos = np.linspace(0, 5, 300)

# Família de exponenciais
base_list = [(np.e, 'blue', '-'), (2, 'green', '--'), (0.5, 'red', ':'), (3, 'orange', '-.')]
for base, c, ls in base_list:
    axes[0].plot(x, base**x, color=c, ls=ls, lw=2,
                 label=f'base={base:.2f}' if base != np.e else 'eˣ (natural)')
axes[0].axhline(1, color='gray', lw=0.8)
axes[0].set_ylim(-0.5, 10); axes[0].set_title('Família de Funções Exponenciais')
axes[0].legend(fontsize=9); axes[0].set_xlabel('x')

# Exponencial negativa (decaimento)
rates = [0.5, 1.0, 2.0]
colors_decay = ['blue', 'green', 'red']
for lam, c in zip(rates, colors_decay):
    axes[1].plot(x_pos, np.exp(-lam * x_pos), color=c, lw=2.5,
                 label=f'λ={lam}: e^(-{lam}t)')
axes[1].set_title('Decaimento Exponencial (Learning Rate Scheduling)')
axes[1].set_xlabel('Época de treino'); axes[1].set_ylabel('Learning Rate')
axes[1].legend(); axes[1].set_ylim(0, 1.1)

# Overflow: problema prático
x_large = np.array([1, 10, 100, 710, 711, 800])
print('Demonstração de overflow numérico:')
for xi in x_large:
    try:
        val = np.exp(xi)
        print(f'  exp({xi:4d}) = {val:.3e}')
    except:
        print(f'  exp({xi:4d}) = ERRO')

axes[2].plot(np.linspace(0, 500, 200), np.exp(np.linspace(0, 500, 200)), 'b-', lw=2)
axes[2].set_title('Crescimento Explosivo de eˣ (risco de overflow em softmax!)')
axes[2].set_xlabel('x'); axes[2].set_ylabel('eˣ')
axes[2].set_yscale('log')

plt.tight_layout()
plt.show()


### Aplicação 1: Função Softmax

A **Softmax** converte um vetor de scores (logits) em probabilidades:

$$\text{Softmax}(z_i) = \frac{e^{z_i}}{\sum_{j} e^{z_j}}$$

É usada na última camada de redes neurais para **classificação multiclasse**.

**Problema numérico:** se $z_i$ é muito grande, $e^{z_i}$ causa overflow.
**Solução (Log-Sum-Exp trick):** subtrair $\max(z)$ antes de calcular.

In [ ]:
# Implementação do Softmax com estabilidade numérica
def softmax_naive(z):
    # Softmax sem estabilidade numérica -- pode causar overflow!
    exp_z = np.exp(z)
    return exp_z / np.sum(exp_z)

def softmax_stable(z):
    # Softmax com Log-Sum-Exp trick -- estável numericamente.
    z_shifted = z - np.max(z)  # subtrai o máximo → nunca overflow
    exp_z = np.exp(z_shifted)
    return exp_z / np.sum(exp_z)

# Testando com valores normais
logits_normal = np.array([2.0, 1.0, 0.5, -1.0])
probs_naive  = softmax_naive(logits_normal)
probs_stable = softmax_stable(logits_normal)

print("Logits normais:", logits_normal)
print(f"Softmax naïve:  {probs_naive}  | soma={probs_naive.sum():.4f}")
print(f"Softmax estável:{probs_stable} | soma={probs_stable.sum():.4f}")

# Testando com valores grandes (overflow!)
logits_large = np.array([1000.0, 999.0, 998.0, 900.0])
print("\nLogits grandes:", logits_large)
try:
    print(f"Softmax naïve:  {softmax_naive(logits_large)}")
except:
    print("Softmax naïve:  OVERFLOW (NaN)")
print(f"Softmax estável:{softmax_stable(logits_large)}")

# Visualização: o que softmax faz
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
classes = ['Gato', 'Cachorro', 'Passáro', 'Peixe']
colors_bar = ['steelblue', 'coral', 'green', 'orange']

axes[0].bar(classes, logits_normal, color=colors_bar)
axes[0].set_title('Logits (scores brutos)'); axes[0].set_ylabel('Score')

axes[1].bar(classes, probs_stable, color=colors_bar)
axes[1].set_title('Probabilidades após Softmax')
axes[1].set_ylabel('Probabilidade')
for i, p in enumerate(probs_stable):
    axes[1].text(i, p + 0.01, f'{p:.2%}', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

### Erro: Overflow em Cross-Entropy

**Erro 1: Calcular diretamente quando probabilidades são muito pequenas**
```python
# ERRADO: quando p ≈ 0, log(p) → -∞
loss = -np.log(pred_probs)  # pred_probs = [0.001, 0.0001, 0.001]

# CORRETO: clip ou usar formulação estável
epsilon = 1e-7
loss = -np.log(np.clip(pred_probs, epsilon, 1.0))
# Ou melhor: usar sparse categorical crossentropy que é numericamente estável
```

**Por que:** log(0) = -∞; números muito pequenos causam -inf ou overflow. Sempre use epsilon ou formulas estáveis.

### Por que em ML: Logs Transformam Multiplicação

1. **MLE e likelihood:** log(a·b·c) = log(a) + log(b) + log(c); diferenciação e soma são mais fáceis
2. **Perda numérica estável:** log de probabilidades pequenas não fica tão negativo quanto produtos
3. **Informação teórica:** Bits e nats usam log; entropia, KL divergence, mutualinformação todos usam log
4. **Otimização:** Minimizar -log(p) é equivalente a maximizar p, mas numericamente mais estável

### Erro: Softmax Ingênua

**Erro 1: Não usar log-sum-exp**
```python
# ERRADO: softmax_naive com logits grandes
logits = np.array([1000, 1001, 999])
bad = np.exp(logits) / np.sum(np.exp(logits))  # → [nan, nan, nan]

# CORRETO: subtrai max antes
logits_stable = logits - np.max(logits)  # [−1, 0, −2]
good = np.exp(logits_stable) / np.sum(np.exp(logits_stable))  # → [0.09, 0.82, 0.09]
```

**Por que:** exp(1000) em float64 já é infinito. Subtrair max mantém números gerenciáveis e resultado idêntico.

### O que observar

1. **Exponenciais crescem rápido:** exp(700) já vira infinity em float64
2. **Log-sum-exp trick:** Subtrair max antes de exp() preserva a proporção mas evita overflow
3. **Softmax instável:** Logits muito grandes causam NaN; estável com log-sum-exp
4. **Learning rate decay:** Exponencial com λ pequeno = decaimento suave; λ grande = decaimento rápido


### Aplicação 2: Learning Rate Decay

Em treinamento de redes neurais, a taxa de aprendizado geralmente **decai** ao longo
do tempo para garantir convergência:

$$\eta_t = \eta_0 \cdot e^{-\lambda t}$$

onde $\eta_0$ é a taxa inicial e $\lambda$ é o fator de decaimento.

### O que concluir: Exponencial em Treinamento

- **Decaimento exponencial funciona:** Reduzir learning rate exponencialmente permite convergência mesmo em platôs
- **Softmax amplifica:** Exponenciais transformam diferenças pequenas em grandes mudanças de probabilidade
- **Numericamente perigoso:** exp(x) fica infinito rápido; sempre use log-sum-exp trick para estabilidade

In [ ]:
# Learning Rate Scheduling: decaimento exponencial
epochs = np.arange(0, 100)

# Diferentes estratégias de decay
lr_constant   = 0.1 * np.ones_like(epochs, dtype=float)
lr_exp_slow   = 0.1 * np.exp(-0.02 * epochs)
lr_exp_fast   = 0.1 * np.exp(-0.05 * epochs)
lr_step       = 0.1 * (0.1 ** (epochs // 30))  # step decay
lr_cosine     = 0.1 * 0.5 * (1 + np.cos(np.pi * epochs / 100))  # cosine annealing

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(epochs, lr_constant,  'k--',  lw=1.5, label='Constante (0.1)')
ax.plot(epochs, lr_exp_slow,  'b-',   lw=2, label='Exp decay λ=0.02')
ax.plot(epochs, lr_exp_fast,  'r-',   lw=2, label='Exp decay λ=0.05')
ax.plot(epochs, lr_step,      'g-',   lw=2, label='Step decay (÷10 a cada 30 épocas)')
ax.plot(epochs, lr_cosine,    'm-',   lw=2, label='Cosine annealing')
ax.set_xlabel('Época'); ax.set_ylabel('Learning Rate (η)')
ax.set_title('Estratégias de Learning Rate Scheduling', fontsize=12)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()
print("Cosine annealing é muito popular em treinamento de LLMs e vision transformers!")

### Por que em ML: Logs Transformam Multiplicação

1. **MLE e likelihood:** log(a·b·c) = log(a) + log(b) + log(c); diferenciação e soma são mais fáceis
2. **Perda numérica estável:** log de probabilidades pequenas não fica tão negativo quanto produtos
3. **Informação teórica:** Bits e nats usam log; entropia, KL divergence, mutualinformação todos usam log
4. **Otimização:** Minimizar -log(p) é equivalente a maximizar p, mas numericamente mais estável

### Conexao com Estatística

- **MLE (Maximum Likelihood):** Transforma produto em soma de logs (log-likelihood)
- **Informação (bits):** Entropia de Shannon usa log₂; KL divergence usa log para comparar distribuições
- **Modelo probabilístico:** Toda densidade tem log; precisa otimizar log-likelihood

### Por que em ML: Entropia e Informação

1. **Perda de classificação:** Cross-entropy é essencialmente -log(probabilidade correta) = penaliza confiança errada
2. **Entropia máxima:** Distribuição uniforme tem máxima entropia; model bem calibrado tem baixa entropia
3. **KL divergence:** Mede divergência entre distribuição verdadeira e predita; usado em VAE, GAN
4. **Princípio máximo de entropia:** Distribuição com máxima entrância subject a constraints é a menos viesada

---
## 7. Funções Logarítmicas → Cross-Entropy e Entropia <a id='7-funcoes-logaritmicas'></a>

O logaritmo é a **função inversa** da exponencial:

$$y = \log_b(x) \iff b^y = x$$

Em ML, usamos principalmente:
- $\ln(x) = \log_e(x)$: logaritmo natural
- $\log_2(x)$: usado em Teoria da Informação (bits)
- $\log_{10}(x)$: escalas de visualização

### Propriedades Essenciais

$$\ln(x \cdot y) = \ln(x) + \ln(y)$$
$$\ln\left(\frac{x}{y}\right) = \ln(x) - \ln(y)$$
$$\ln(x^a) = a \cdot \ln(x)$$

> **A propriedade mais importante:** transforma **produtos** em **somas**.
> Isso é fundamental para converter o produto de probabilidades (verossimilhança)
> em uma soma (log-verossimilhança), que é numericamente estável e fácil de derivar.

In [ ]:
# Função logarítmica: bases e comportamento
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

x = np.linspace(0.01, 8, 400)

# Diferentes bases
for base, c, lbl in [(np.e, 'blue', 'ln(x) -- natural'),
                     (2, 'green', 'log₂(x) -- bits'),
                     (10, 'orange', 'log₁₀(x) -- décadas'),
                     (0.5, 'red', 'log₀.₅(x) -- decrescente')]:
    if base == 0.5:
        axes[0].plot(x, np.log(x)/np.log(base), color=c, lw=2, ls='--', label=lbl)
    else:
        axes[0].plot(x, np.log(x)/np.log(base), color=c, lw=2, label=lbl)
axes[0].axhline(0, color='gray', lw=0.8); axes[0].axvline(1, color='gray', lw=0.8)
axes[0].set_ylim(-3, 3); axes[0].set_title('Logaritmos em Diferentes Bases')
axes[0].legend(fontsize=9); axes[0].set_xlabel('x')
axes[0].annotate('ln(1)=0 para qualquer base', xy=(1, 0), xytext=(2.5, -1.5),
                 arrowprops=dict(arrowstyle='->', color='gray'))

# Relação log-exp (funções inversas)
x2 = np.linspace(0.01, 3, 200)
axes[1].plot(x2, np.exp(x2), 'b-', lw=2, label='eˣ')
axes[1].plot(x2, np.log(x2), 'r-', lw=2, label='ln(x)')
axes[1].plot(x2, x2, 'k--', lw=1, label='y = x')
axes[1].set_xlim(0, 3); axes[1].set_ylim(-2, 5)
axes[1].set_title('eˣ e ln(x): Funções Inversas')
axes[1].legend()
axes[1].fill_between(x2, np.exp(x2), x2, alpha=0.1, color='blue')
axes[1].fill_between(x2, np.log(x2), x2, alpha=0.1, color='red')

# ln(x) próximo de 0 → -∞ (instabilidade numérica)
x3 = np.linspace(1e-5, 1, 300)
axes[2].plot(x3, np.log(x3), 'purple', lw=2.5)
axes[2].axhline(0, color='gray', lw=0.8)
axes[2].set_title('ln(x) próximo de 0 → -∞ (Risco em cross-entropy!)')
axes[2].set_xlabel('x (probabilidade prevista)'); axes[2].set_ylabel('ln(x)')
axes[2].annotate('Se modelo prevê p≈0 para a classe correta → perda enorme!',
                 xy=(0.05, -3), xytext=(0.3, -4),
                 arrowprops=dict(arrowstyle='->', color='red'), color='red')

plt.tight_layout()
plt.show()


### Conexao com Redes Neurais

- **Camada oculta:** ReLU ou Sigmoid transforma inputs linearmente combinados em não-linear
- **Backpropagation:** Derivada de ativação multiplica a regra da cadeia
- **Escolha importa:** Sigmoid/Tanh sofrem vanishing gradient; ReLU/GELU são modernos

### Aplicação 1: Binary Cross-Entropy (Log Loss)

A função de perda mais usada em classificação binária é:

$$\mathcal{L}(y, \hat{p}) = -\frac{1}{n} \sum_{i=1}^{n} \left[ y_i \log(\hat{p}_i) + (1 - y_i) \log(1 - \hat{p}_i) \right]$$

onde:
- $y_i \in \{0, 1\}$: classe real
- $\hat{p}_i \in (0, 1)$: probabilidade prevista pelo modelo

---
## 8. Sigmoid e Funcoes de Ativacao -> Redes Neurais <a id='8-sigmoid-ativacoes'></a>

### Intuicao: O Interruptor Suave

Pense em um termostato: quando a temperatura esta muito abaixo do limite, ele esta "desligado" (0). Quando esta muito acima, esta "ligado" (1). E na zona intermediaria, ele transiciona suavemente. A sigmoid faz exatamente isso -- transforma qualquer numero real em um valor entre 0 e 1, criando uma transicao suave.

Em ML, isso e util para:
- **Classificacao binaria:** Transformar scores brutos em probabilidades ("70% chance de ser spam")
- **Gates em LSTMs:** Controlar quanto de informacao passa (0 = bloqueia tudo, 1 = passa tudo)

### Definicao Formal

$$\sigma(x) = \frac{1}{1 + e^{-x}} = \frac{e^x}{1 + e^x}$$

### Propriedades da Sigmoid

| Propriedade | Valor | Importancia |
|-------------|-------|-------------|
| Dominio | $(-\infty, +\infty)$ | Aceita qualquer valor real |
| Imagem | $(0, 1)$ | Saida interpretavel como probabilidade |
| Ponto medio | $\sigma(0) = 0.5$ | Threshold natural de decisao |
| Derivada | $\sigma'(x) = \sigma(x)(1-\sigma(x))$ | Elegante, usada no backpropagation |
| Saturacao | $\sigma(x) \to 0$ ou $1$ para $|x|$ grande | Causa **vanishing gradient**! |

### Por que a Sigmoid Causa Vanishing Gradient?

A derivada maxima da sigmoid e apenas 0.25 (em x=0). Em redes profundas, backpropagation multiplica derivadas camada por camada (regra da cadeia). Se cada camada multiplica por no maximo 0.25, apos 10 camadas o gradiente e $0.25^{10} \approx 10^{-6}$ -- praticamente zero. As primeiras camadas da rede "nao aprendem" porque o sinal de erro nao chega ate elas.

**Solucao historica:** Substituir sigmoid por **ReLU** ($\max(0, x)$), cuja derivada e 1 para x > 0. Isso preserva o gradiente ao longo das camadas.

### Funcoes de Ativacao em Deep Learning

As funcoes de ativacao introduzem **nao-linearidade** nas redes neurais.
Sem elas, qualquer rede profunda seria equivalente a uma simples regressao linear!

Por que? Porque a composicao de funcoes lineares e ainda linear: $f(g(x)) = f(Ax+b) = A'x + b'$. Sem a nao-linearidade, empilhar 100 camadas nao traria nenhum beneficio sobre uma unica camada.

### Aplicação 2: Entropia da Informação

A **Entropia de Shannon** mede a "quantidade de informação" (ou incerteza) em uma
distribuição de probabilidade:

$$H(p) = -\sum_{i} p_i \log_2(p_i)$$

- Máxima entropia: distribuição **uniforme** (máxima incerteza)
- Entropia zero: distribuição **determinística** (certeza total)

> Em **árvores de decisão**, o critério de entropia (ou Gini, que é similar)
> guia os splits: escolhe a divisão que **maximiza a redução de entropia**.

In [ ]:
# Entropia de Shannon: intuição visual
def entropy(probs, base=2):
    # Calcula entropia de Shannon.
    probs = np.array(probs)
    probs = probs[probs > 0]  # evitar log(0)
    return -np.sum(probs * np.log(probs) / np.log(base))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Entropia binária (2 classes)
p_range = np.linspace(0.001, 0.999, 300)
h_binary = -(p_range * np.log2(p_range) + (1-p_range) * np.log2(1-p_range))

axes[0].plot(p_range, h_binary, 'b-', lw=2.5)
axes[0].axvline(0.5, color='r', ls='--', lw=1.5, label='Máxima incerteza (p=0.5)')
axes[0].scatter([0.5], [1.0], color='r', s=80, zorder=5)
axes[0].set_xlabel('Probabilidade da Classe 1'); axes[0].set_ylabel('Entropia H(p) [bits]')
axes[0].set_title('Entropia Binária (distribuição de 2 classes)')
axes[0].legend()

# Exemplos concretos
examples = [
    ('Determinístico\n[1.0, 0.0, 0.0]', [1.0, 0.0, 0.0]),
    ('Pouco incerto\n[0.7, 0.2, 0.1]', [0.7, 0.2, 0.1]),
    ('Muito incerto\n[0.4, 0.35, 0.25]', [0.4, 0.35, 0.25]),
    ('Uniforme\n[⅓, ⅓, ⅓]', [1/3, 1/3, 1/3]),
]

entropies = [entropy(e[1]) for e in examples]
labels = [e[0] for e in examples]
bar_colors = plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, len(examples)))
bars = axes[1].bar(range(len(examples)), entropies, color=bar_colors)
axes[1].set_xticks(range(len(examples))); axes[1].set_xticklabels(labels, fontsize=8)
axes[1].set_ylabel('Entropia [bits]')
axes[1].set_title('Entropia de Distribuições de 3 Classes')
for bar, h in zip(bars, entropies):
    axes[1].text(bar.get_x() + bar.get_width()/2, h + 0.01,
                 f'{h:.3f}', ha='center', fontsize=9)

# KL Divergence (prévia)
p_true = np.array([0.6, 0.3, 0.1])  # distribuição real
q_approx = np.array([0.4, 0.4, 0.2])  # distribuição aproximada
kl_div = np.sum(p_true * np.log(p_true / q_approx))
axes[2].bar(['P (real)', 'Q (aprox)'],
            [entropy(p_true), entropy(q_approx)], color=['steelblue', 'coral'])
axes[2].set_title(f'Entropia de P vs Q\nKL(P||Q) = {kl_div:.3f}')
axes[2].set_ylabel('Entropia [bits]')

plt.tight_layout()
plt.show()


---
## 8. Sigmoid e Funções de Ativação → Redes Neurais <a id='8-sigmoid-ativacoes'></a>

A função **Sigmoid** (ou logística) combina as funções exponencial e logarítmica:

$$\sigma(x) = \frac{1}{1 + e^{-x}} = \frac{e^x}{1 + e^x}$$

### Propriedades da Sigmoid

| Propriedade | Valor | Importância |
|-------------|-------|-------------|
| Domínio | $(-\infty, +\infty)$ | Aceita qualquer valor real |
| Imagem | $(0, 1)$ | Saída interpretável como probabilidade |
| Ponto médio | $\sigma(0) = 0.5$ | Threshold natural de decisão |
| Derivada | $\sigma'(x) = \sigma(x)(1-\sigma(x))$ | Elegante, usada no backpropagation |
| Saturação | $\sigma(x) \to 0$ ou $1$ para $|x|$ grande | Causa **vanishing gradient**! |

### Funções de Ativação em Deep Learning

As funções de ativação introduzem **não-linearidade** nas redes neurais.
Sem elas, qualquer rede profunda seria equivalente a uma simples regressão linear!

### O que concluir: Síntese das Funções

- **Cada função tem seu propósito:** Lineares para combinação, exponencial para softmax, log para perda
- **Composição = expressividade:** Combinar funções permite modelar qualquer relação (universal approximation)
- **Estabilidade numérica é prática:** Teoria perfeita falha com overflow/underflow; sempre use truques

In [ ]:
# Sigmoid: implementação e propriedades
def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

def sigmoid_derivative(x):
    s = sigmoid(x)
    return s * (1 - s)

x = np.linspace(-6, 6, 300)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Sigmoid e sua derivada
axes[0].plot(x, sigmoid(x), 'b-', lw=2.5, label='σ(x)')
axes[0].plot(x, sigmoid_derivative(x), 'r-', lw=2.5,
             label="σ'(x) = σ(x)·(1-σ(x))")
axes[0].axhline(0.5, color='gray', ls=':', lw=1.5)
axes[0].axhline(0.25, color='red', ls=':', lw=1, alpha=0.5, label="Máx. σ'(x)=0.25 em x=0")
axes[0].axvline(0, color='gray', ls=':', lw=1.5)
axes[0].set_title('Sigmoid e sua Derivada')
axes[0].legend(fontsize=9); axes[0].set_xlabel('x')

# Vanishing gradient problem
deep_layers = np.arange(0, 10)
grad_sigmoid = 0.25**deep_layers  # máx derivada da sigmoid = 0.25
axes[1].semilogy(deep_layers, grad_sigmoid, 'r-o', lw=2, ms=6)
axes[1].set_title('Vanishing Gradient na Sigmoid (cada camada multiplica por ≤0.25)')
axes[1].set_xlabel('Número de camadas'); axes[1].set_ylabel('Magnitude do gradiente (log)')
axes[1].annotate(f'10 camadas: gradiente = {0.25**10:.2e}!',
                 xy=(10, 0.25**10), xytext=(5, 1e-4),
                 arrowprops=dict(arrowstyle='->'))

# Comparação de ativações
def tanh(x): return np.tanh(x)
def relu(x): return np.maximum(0, x)
def leaky_relu(x, alpha=0.01): return np.where(x >= 0, x, alpha*x)
def gelu(x): return x * sigmoid(1.702 * x)  # approx

for fn, lbl, c in [(sigmoid, 'Sigmoid', 'blue'), (tanh, 'Tanh', 'green'),
                   (relu, 'ReLU', 'red'), (leaky_relu, 'Leaky ReLU', 'orange'),
                   (gelu, 'GELU', 'purple')]:
    axes[2].plot(x, fn(x), lw=2, label=lbl, color=c)
axes[2].axhline(0, color='gray', lw=0.8)
axes[2].set_ylim(-2, 4); axes[2].set_title('Comparação de Funções de Ativação')
axes[2].legend(fontsize=9); axes[2].set_xlabel('x')

plt.tight_layout()
plt.show()


### O que concluir: Síntese das Funções

- **Cada função tem seu propósito:** Lineares para combinação, exponencial para softmax, log para perda
- **Composição = expressividade:** Combinar funções permite modelar qualquer relação (universal approximation)
- **Estabilidade numérica é prática:** Teoria perfeita falha com overflow/underflow; sempre use truques

### Por que em ML: Funções de Ativação

1. **Sem ativação = tudo é linear:** Composição de lineares é só uma grande linear; não consegue aprender padrões
2. **ReLU é rápido:** max(0, x) é muito mais rápido que sigmoid; escalável para redes gigantes
3. **Diferenciabilidade:** Exceto em pontos isolados, ativações são diferenciáveis; permite backprop
4. **Gradientes transmitem info:** Ativação com bom gradiente permite que sinal viaje através de muitas camadas

### O que observar: Sigmoid e suas Limitações

- **Satura em extremos:** Quando x < -5 ou x > 5, a derivada ≈ 0 (vanishing gradient)
- **Saída em (0, 1):** Perfeito para probabilidades, mas não para representar valores negativos
- **S-curve suave:** Transição suave de 0 → 1 permite gradientes em toda a faixa

In [ ]:
# Tabela comparativa de ativações
print("=" * 75)
print(f"{'Ativação':<12} {'Imagem':<18} {'Derivada Máx':<16} {'Uso Principal'}")
print("=" * 75)
activations_info = [
    ("Sigmoid",    "(0, 1)",        "0.25 em x=0",  "Classificação binária (saída)"),
    ("Tanh",       "(-1, 1)",       "1.0 em x=0",   "RNNs, LSTM (vanishing tbm)"),
    ("ReLU",       "[0, +∞)",       "1.0 (x>0)",    "Redes profundas (padrão)"),
    ("Leaky ReLU", "(-∞, +∞)",      "1.0 ou 0.01",  "Evita neurônios mortos"),
    ("GELU",       "(-0.17, +∞)",   "~1.1",         "BERT, GPT, Transformers"),
    ("Softmax",    "(0,1) somando1","--",             "Classificação multiclasse"),
]
for name, img, deriv, use in activations_info:
    print(f"{name:<12} {img:<18} {deriv:<16} {use}")
print("=" * 75)

# Conexão Sigmoid → Logistic Regression (implementação manual)
print("\n── Logistic Regression: a sigmoid na prática ──────────────────────────")

# Gerar dados de classificação sintéticos
np.random.seed(42)
n_samples = 200
X_cls = np.random.randn(n_samples, 1) * 2
y_cls = (X_cls.flatten() + np.random.randn(n_samples) * 0.5) > 0
y_cls = y_cls.astype(float)

# Treinar regressão logística manualmente (gradient descent)
def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

w = 0.1
b = 0.0
learning_rate = 0.1
n_iterations = 1000

for _ in range(n_iterations):
    z = w * X_cls.flatten() + b
    probs = sigmoid(z)
    dw = -np.mean((y_cls - probs) * X_cls.flatten())
    db = -np.mean(y_cls - probs)
    w -= learning_rate * dw
    b -= learning_rate * db

print(f"Peso aprendido: w={w:.4f}, b={b:.4f}")
print(f"A saída do modelo é: σ({w:.2f}·x + {b:.2f})")
print("→ exatamente a função sigmoid aplicada a uma combinação linear!")


---
## 10. Exercicios Praticos <a id='10-exercicios'></a>

Coloque em pratica o que voce aprendeu. Para cada exercicio:
1. **Leia o enunciado** e pense na abordagem antes de escrever codigo
2. **Tente implementar** na celula de pratica (marcada com "Sua solucao aqui")
3. **Compare com a solucao** na celula seguinte (so olhe depois de tentar!)

### Exercicio 1 -- Funcoes Lineares

**Tarefa:** Implemente uma funcao `linear_predict(X, w, b)` que calcula $\hat{y} = wX + b$.
Depois, implemente `mse(y_true, y_pred)` que calcula o erro quadratico medio.

**Objetivo:** Comparar o MSE usando os parametros corretos (w=2.5, b=1.0) vs parametros errados (w=1.0, b=5.0).

**Dica:** Use operacoes vetorizadas do NumPy -- nao precisa de loops.

### Conexao com Machine Learning: Ativações

- **Sigmoid:** Usada em última camada de classificação binária (gera probabilidade)
- **Softmax:** Generalização de sigmoid para multiclasse (gera distribuição)
- **ReLU:** Padrão em redes convolucionais e transformers (computacionalmente eficiente)
- **Composição:** f(g(h(x))) é como redes profundas transformam inputs em outputs

In [ ]:
# ── Exercicio 1: Sua solucao aqui ──────────────────────────────────────────
np.random.seed(0)
X_ex1 = np.random.uniform(0, 10, 50)
y_ex1 = 2.5 * X_ex1 + 1.0 + np.random.normal(0, 1.5, 50)  # dados com ruido

def linear_predict(X, w, b):
    """Calcula predicoes lineares: y_hat = w*X + b"""
    pass  # Implemente aqui

def mse(y_true, y_pred):
    """Calcula o erro quadratico medio"""
    pass  # Implemente aqui

# Teste suas funcoes:
# y_hat_bom = linear_predict(X_ex1, 2.5, 1.0)
# y_hat_ruim = linear_predict(X_ex1, 1.0, 5.0)
# print(f"MSE (parametros bons):  {mse(y_ex1, y_hat_bom):.4f}")
# print(f"MSE (parametros ruins): {mse(y_ex1, y_hat_ruim):.4f}")

In [ ]:
# ── Exercicio 1: SOLUCAO ──────────────────────────────────────────────────
def linear_predict(X, w, b):
    return w * X + b

def mse(y_true, y_pred):
    return np.mean((y_true - y_pred)**2)

y_hat_bom = linear_predict(X_ex1, 2.5, 1.0)
y_hat_ruim = linear_predict(X_ex1, 1.0, 5.0)

print(f"MSE (parametros bons w=2.5, b=1.0):  {mse(y_ex1, y_hat_bom):.4f}")
print(f"MSE (parametros ruins w=1.0, b=5.0): {mse(y_ex1, y_hat_ruim):.4f}")
print(f"\nO MSE com parametros bons e ~{mse(y_ex1, y_hat_ruim)/mse(y_ex1, y_hat_bom):.0f}x menor!")
print("Isso porque w=2.5 e b=1.0 sao os parametros que geraram os dados.")

In [ ]:
# ── Exercicio 2: TAREFA DO ALUNO ──────────────────────────────────────────
z_ex2 = np.array([3.0, 1.0, 0.2])
classes_ex2 = ['Gato', 'Cachorro', 'Hamster']

# Implemente a softmax estavel (com log-sum-exp trick) e aplique a z_ex2
# Depois, identifique qual classe o modelo escolheria e com qual confianca

def softmax_ex(z):
    """Softmax com estabilidade numerica"""
    pass  # Implemente aqui

# probs = softmax_ex(z_ex2)
# print("Probabilidades:", {c: f'{p:.2%}' for c, p in zip(classes_ex2, probs)})
# print(f"Classe escolhida: {classes_ex2[np.argmax(probs)]}")

In [ ]:
# ── Exercicio 2: SOLUCAO ──────────────────────────────────────────────────
def softmax_ex(z):
    z_shifted = z - np.max(z)  # log-sum-exp trick
    exp_z = np.exp(z_shifted)
    return exp_z / np.sum(exp_z)

probs = softmax_ex(z_ex2)
print("Probabilidades:")
for c, p in zip(classes_ex2, probs):
    print(f"  {c}: {p:.2%}")
print(f"\nClasse escolhida: {classes_ex2[np.argmax(probs)]} ({probs.max():.2%} de confianca)")
print(f"Soma das probabilidades: {probs.sum():.4f} (deve ser 1.0)")
print(f"\nObserve como o logit 3.0 (Gato) domina: a exponencial amplifica")
print(f"a diferenca entre 3.0 e 1.0, tornando Gato ~7x mais provavel que Cachorro.")

In [ ]:
# ── Exercicio 3: TAREFA DO ALUNO ──────────────────────────────────────────
scenarios_ex3 = [
    ('A: Correto e confiante', 1, 0.90),
    ('B: Correto e incerto',   1, 0.50),
    ('C: Errado e confiante',  1, 0.10),
    ('D: Correto (classe 0)',  0, 0.10),
]

# Para cada cenario, calcule: -[y*log(p) + (1-y)*log(1-p)]
# Interprete: por que a perda do cenario C e tao alta?
# Dica: pense no grafico de -log(p) que vimos na secao 7

# TAREFA DO ALUNO: calcule a BCE para cada cenario e imprima os resultados.

In [ ]:
# ── Exercicio 3: SOLUCAO ──────────────────────────────────────────────────
scenarios_ex3 = [
    ('A', 1, 0.90),
    ('B', 1, 0.50),
    ('C', 1, 0.10),
    ('D', 0, 0.10),
]

eps = 1e-15
print(f"{'Cenario':<25} {'y':>3} {'p_hat':>6} {'BCE':>8}  Interpretacao")
print("-" * 75)
for name, y, p in scenarios_ex3:
    bce = -(y * np.log(p + eps) + (1-y) * np.log(1 - p + eps))
    print(f"{name:<25} {y:>3} {p:>6.2f} {bce:>8.4f}  ", end="")
    if bce < 0.2:
        print("Perda baixa -- modelo acertou com confianca")
    elif bce < 1.0:
        print("Perda moderada -- modelo incerto")
    else:
        print("Perda ALTA -- modelo errou com confianca!")

print(f"\nPor que C tem perda tao alta?")
print(f"Porque -log(0.10) = {-np.log(0.10):.2f}. O modelo disse '90% de chance de NAO")
print(f"ser classe 1', mas era classe 1. A cross-entropy penaliza exponencialmente")
print(f"previsoes confiantes e erradas -- isso forca o modelo a ser calibrado.")


In [ ]:
# ── Exercicio 4: TAREFA DO ALUNO ──────────────────────────────────────────
def f(w):
    return (w - 3)**2 + 2

def df(w):
    return 2 * (w - 3)

# Implemente gradient descent:
# 1. Comece em w = 10.0
# 2. Use learning_rate = 0.1
# 3. Execute 50 iteracoes
# 4. A cada iteracao: w = w - learning_rate * df(w)
# 5. Guarde o historico de w e f(w)

w_init = 10.0
learning_rate = 0.1
n_iterations = 50

# TAREFA DO ALUNO: implemente o loop de gradient descent abaixo.

---
## Erros Comuns e Armadilhas

Antes de seguir para o proximo notebook, fique atento a estes erros frequentes:

**1. Overflow na Softmax**
- Errado: `np.exp(logits) / np.sum(np.exp(logits))`
- Certo: `np.exp(logits - max(logits)) / np.sum(np.exp(logits - max(logits)))`
- Sintoma: resultados `nan` ou `inf` quando logits sao grandes

**2. Log de zero na Cross-Entropy**
- Errado: `np.log(p)` quando p pode ser 0
- Certo: `np.log(np.clip(p, 1e-15, 1.0))`
- Sintoma: `-inf` na loss, que corrompe todo o treinamento

**3. Confundir inclinacao com intercepto**
- A inclinacao (peso) diz "quanto cada feature importa"
- O intercepto (bias) diz "qual o baseline sem nenhuma feature"
- Esquecer o bias limita o modelo a passar pela origem

**4. Grau polinomial alto demais**
- Grau 1 (underfitting): nao captura relacoes curvilineas
- Grau ideal: captura o padrao real dos dados
- Grau muito alto (overfitting): memoriza ruido, performa mal em dados novos
- Regra pratica: use validacao cruzada para escolher o grau

---

## Resumo: Conectando os Conceitos

Neste notebook, voce aprendeu as funcoes matematicas fundamentais para ML:

```
Funcoes Lineares  -->  Funcoes Polinomiais  -->  Funcoes Exponenciais
    |                       |                         |
    v                       v                         v
Regressao Linear     Features Polinomiais        Softmax
Camada Densa         Aproximacao de curvas        Learning Rate Decay
    |                       |                         |
    +----------+------------+-------+-----------------+
               |                    |
               v                    v
         Funcoes de Custo     Funcoes de Ativacao
         (MSE, Cross-Entropy)  (Sigmoid, ReLU, GELU)
               |                    |
               +--------+-----------+
                        |
                        v
                  Treinamento do Modelo
                  (Gradient Descent -- notebook 0.4 e 0.8)
```

| Funcao | Formula | Aplicacao em ML |
|--------|---------|-----------------|
| **Linear** | $f(x) = mx + b$ | Regressao linear, camada densa |
| **Polinomial** | $f(x) = \sum a_i x^i$ | Feature engineering, aproximacao |
| **Exponencial** | $f(x) = e^x$ | Softmax, decaimento LR |
| **Logaritmica** | $f(x) = \ln(x)$ | Cross-entropy, log-likelihood |
| **Sigmoid** | $\sigma(x) = 1/(1+e^{-x})$ | Classificacao binaria, ativacao |
| **Somatorio** | $\sum f(x_i)$ | Todas as funcoes de custo |
| **Produtorio** | $\prod P(x_i)$ | Verossimilhanca (MLE) |

### Pre-requisitos para os Proximos Notebooks

| Proximo Notebook | O que voce precisa daqui | Secao de referencia |
|------------------|--------------------------|---------------------|
| 0.2 -- Vetores | Funcoes lineares, produto escalar | Secao 4 |
| 0.3 -- Matrizes | Operacoes algebricas | Secao 2 |
| 0.4 -- Derivadas | Conceito de inclinacao, sigmoid | Secoes 4 e 8 |
| 0.6 -- Probabilidade | Logaritmos, produtorios | Secoes 7 e 9 |
| 3.1 -- Classificacao | Sigmoid, softmax, cross-entropy | Secoes 6, 7 e 8 |

### Proximos Passos

- **Notebook 0.2:** Algebra Linear I -- Vetores e Operacoes
  - Produto interno, normas, similaridade de cosseno
  - Aplicacoes: KNN, embeddings, atencao em Transformers

- **Notebook 0.3:** Algebra Linear II -- Matrizes e Transformacoes
  - Autovalores, SVD, decomposicoes
  - Aplicacoes: PCA, compressao, sistemas de recomendacao

---

### Referencias

- **Bishop, C.M.** -- *Pattern Recognition and Machine Learning* (Cap. 1)
- **Goodfellow et al.** -- *Deep Learning* (Cap. 2: Linear Algebra + Cap. 3: Probability)
- **3Blue1Brown** -- Serie *Essence of Calculus* (YouTube)
- **Khan Academy** -- Pre-calculo e funcoes

### Exercício 2 -- Softmax e Probabilidades
Dado o vetor de logits `z = [3.0, 1.0, 0.2]`, calcule manualmente a Softmax
e verifique que as probabilidades somam 1. Em seguida, responda:
qual classe o modelo escolheria? Qual é a "confiança" nessa previsão?

In [ ]:
# Exercício 2: sua solução aqui
z_ex2 = np.array([3.0, 1.0, 0.2])
classes_ex2 = ['Gato', 'Cachorro', 'Hamster']

# TAREFA DO ALUNO: calcule a softmax de z_ex2
# Lembre-se de usar o log-sum-exp trick para estabilidade numérica

# TAREFA DO ALUNO: imprima as probabilidades e a classe escolhida

# Descomentar para ver solução:
# def softmax(z):
#     z = z - z.max()
#     return np.exp(z) / np.exp(z).sum()
# probs = softmax(z_ex2)
# print("Probabilidades:", {c: f'{p:.2%}' for c, p in zip(classes_ex2, probs)})
# print(f"Classe escolhida: {classes_ex2[np.argmax(probs)]} ({probs.max():.2%} de confiança)")

### Exercício 3 -- Cross-Entropy
Calcule a binary cross-entropy para os seguintes cenários e explique
por que as perdas têm os valores que têm:

| Cenário | y_true | ŷ (prob) |
|---------|--------|-----------|
| A | 1 | 0.90 |
| B | 1 | 0.50 |
| C | 1 | 0.10 |
| D | 0 | 0.10 |

In [ ]:
# Exercício 3: sua solução aqui
scenarios_ex3 = [
    ('A', 1, 0.90),
    ('B', 1, 0.50),
    ('C', 1, 0.10),
    ('D', 0, 0.10),
]

# TAREFA DO ALUNO: para cada cenário, calcule -[y*log(p) + (1-y)*log(1-p)]
# e interprete o resultado

# Descomentar para ver solução:
# print(f"{'Cenário':<10} {'y':<5} {'ŷ':<6} {'BCE':<10} Interpretação")
# for name, y, p in scenarios_ex3:
#     bce = -(y * np.log(p + 1e-15) + (1-y) * np.log(1 - p + 1e-15))
#     interp = "Correto e confiante" if (y==1 and p>0.7) or (y==0 and p<0.3) else #              "Errado e confiante"   if (y==1 and p<0.3) or (y==0 and p>0.7) else #              "Incerto"
#     print(f"{name:<10} {y:<5} {p:<6.2f} {bce:<10.4f} {interp}")

### Exercício 4 -- Desafio: Implementando Gradient Descent Manualmente
Implemente gradient descent para encontrar o mínimo da função
$f(w) = (w - 3)^2 + 2$ (que tem mínimo em $w = 3$).

A regra de atualização é: $w_{t+1} = w_t - \eta \cdot f'(w_t)$

onde $f'(w) = 2(w - 3)$

In [ ]:
# Exercício 4: Gradient Descent manual
def f(w):
    return (w - 3)**2 + 2

def df(w):
    return 2 * (w - 3)  # derivada de f

# TAREFA DO ALUNO: implemente o loop de gradient descent
# Parâmetros:
#   w_init = 10.0 (ponto inicial)
#   learning_rate = 0.1
#   n_iterations = 50
# Acompanhe a trajetória de w e f(w) a cada iteração

# Descomentar para ver solução:
# w, lr, n_iter = 10.0, 0.1, 50
# history_w, history_f = [w], [f(w)]
# for _ in range(n_iter):
#     w = w - lr * df(w)
#     history_w.append(w); history_f.append(f(w))
#
# fig, axes = plt.subplots(1, 2, figsize=(12, 4))
#
# w_range = np.linspace(-1, 12, 200)
# axes[0].plot(w_range, f(w_range), 'b-', lw=2.5)
# axes[0].scatter(history_w[::5], f(np.array(history_w[::5])), c=range(len(history_w[::5])),
#                 cmap='RdYlGn', s=80, zorder=5)
# axes[0].set_title('Trajetória do Gradient Descent em f(w)=(w-3)²+2')
# axes[0].set_xlabel('w'); axes[0].set_ylabel('f(w)')
#
# axes[1].plot(history_f, 'r-o', ms=4, lw=1.5)
# axes[1].axhline(2, ls='--', color='k', label='Mínimo = 2.0')
# axes[1].set_title('Convergência do Custo'); axes[1].set_xlabel('Iteração')
# axes[1].legend()
#
# plt.tight_layout(); plt.show()
# print(f"w final: {history_w[-1]:.6f}  (ótimo: 3.0)")
# print(f"f(w) final: {history_f[-1]:.6f} (ótimo: 2.0)")

## Resumo do Notebook

Você aprendeu as funções matemáticas fundamentais para ML:

| Tipo | Propósito | Exemplo em ML |
|------|-----------|---------------|
| Linear | Regressão simples | y = mx + b |
| Quadrática | Polinomial features | Decisão não-linear |
| Exponencial | Softmax, decaimento | e^z em redes neurais |
| Logarítmica | Cross-entropy, entropia | -log(p) na perda |
| Sigmoid | Ativação, probabilidade | Output binário |

### Hierarquia de Conceitos

1. **Base:** Álgebra e Propriedades (distributiva, potências, logs)
2. **Nível 1:** Funções básicas (linear, quadrática)
3. **Nível 2:** Funções exponenciais e logarítmicas (softmax, cross-entropy)
4. **Nível 3:** Funções de ativação (sigmoid, ReLU, GELU)
5. **Nível 4:** Aplicações (regressão, classificação, redes neurais)

### Checklist de Compreensão

- [ ] Entendo o que é uma função e seus elementos (domínio, imagem)
- [ ] Consigo identificar funções lineares em problemas de ML
- [ ] Sei por que softmax usa exponencial
- [ ] Compreendo o truque de log-sum-exp para estabilidade numérica
- [ ] Conheço o problema de vanishing gradient na sigmoid
- [ ] Posso explicar cross-entropy em termos de logaritmos

### Proximos Passos

1. **Notebook 0.2:** Álgebra Linear (vetores, matrizes, projeção)
2. **Notebook 0.3:** Cálculo (derivadas, gradientes, otimização)
3. **Notebook 0.4:** Probabilidade (distribuições, Bayes)
4. **Notebook 0.5:** Implementar gradient descent com as derivadas aprendidas



### Conexao com Próximos Notebooks

- **0_4_calculo_derivadas.ipynb:** Derivadas de todas essas funções = regras de cadeia em backprop
- **0_3_algebra_linear_matrizes.ipynb:** Vetores/matrizes = maneira eficiente de aplicar funções linearmente
- **0_8_otimizacao_ml.ipynb:** Gradient descent usa derivadas para atualizar pesos iterativamente
- **4_1_fundamentos_redes_neurais.ipynb:** Composição de funções = redes profundas; sem composição seria tudo linear

### O que concluir: Papel de Cada Função

- **Lineares:** Base; criamos features lineares antes de adicionar complexidade
- **Polinomiais:** Capem não-linearidades simples; criamos features polinomiais
- **Exponencial/Log:** Transformam multiplicação em adição; fundamentais em probabilidade
- **Sigmoid/Softmax:** Comprimem para [0,1]; permitem interpretação probabilística
- **Composição:** Redes neurais são composições de ativações; cada camada é uma transformação não-linear

### Por que em ML

1. **Exponenciais crescem rápido:** Softmax usa exp() para amplificar diferenças pequenas entre logits.
2. **Log transforma produto em soma:** MLE (Maximum Likelihood Estimation) usa ln(L) para evitar underflow.
3. **Sigmoid comprime em (0,1):** Perfeito para probabilidades em classificação binária.
4. **ReLU é linear para x>0:** Mais rápido que sigmoid, evita vanishing gradient.
5. **Log-sum-exp trick:** Fundamental para estabilidade numérica em deep learning.
6. **Cross-entropy penaliza confiança:** Erros confiantes têm custo exponencialmente maior.
7. **Composição de funções:** Redes neurais são g(f(h(x))), onde f,g,h são ativações.
8. **Gradientes desaparecem:** Sigmoid tem derivada máxima 0.25, multiplicar muitas camadas → 0.


### Erro: Armadilhas Comuns

**Erro 1: Não usar log-sum-exp trick em softmax**
```python
# ERRADO: pode dar NaN
softmax = np.exp(logits) / np.sum(np.exp(logits))

# CERTO: numericamente estável
logits_shifted = logits - np.max(logits)
softmax = np.exp(logits_shifted) / np.sum(np.exp(logits_shifted))
```

**Erro 2: Tomar log de zero ou número negativo**
```python
# ERRADO: log(0) = -∞, log(-1) = NaN
loss = -np.log(predictions)

# CERTO: adicionar pequeno epsilon
eps = 1e-15
loss = -np.log(predictions + eps)
```

**Erro 3: Usar sigmoid em regressão**
Sigmoid comprime output em (0,1). Regressão precisa de todo R. Use identidade ou ReLU.

**Erro 4: Confundir derivada da sigmoid**
σ'(x) = σ(x)·(1-σ(x)), não σ'(x) = σ(x).

**Erro 5: Esquecer que log é concavo**
log(p) é sempre negativa para 0 < p < 1. Isso inverte desigualdades em algumas provas!


### O que concluir

1. Exponenciais são onipresentes em ML por causa da softmax
2. Log transforma produtos em somas, fundamental para MLE
3. Log-sum-exp é um truque essencial para estabilidade numérica
4. Sem normalização, números grandes causam overflow/underflow


### O que concluir

1. Sigmoid é perfeita para probabilidades mas sofre de vanishing gradient
2. ReLU é mais rápido mas morre para x<0 (problema do neurônio morto)
3. Ativações modernas (GELU, SiLU) combinam velocidade com estabilidade
4. A escolha de ativação afeta dramaticamente o treinamento de redes profundas
5. Composição de funções é como redes neurais funcionam: f(g(h(x)))


### Conexao com Machine Learning

- **Softmax:** Converte scores em probabilidades via exponencial
- **Entropy:** Mede incerteza; máxima em distribuição uniforme
- **Cross-entropy:** Perda padrão para classificação multiclasse
- **KL Divergence:** Distância entre distribuições (usada em VAEs)


### Erro: Problemas Numéricos Comuns

**Erro 1: Overflow em exp()**
exp(1000) = inf, causa NaN em divisão.

**Erro 2: Underflow em log()**
log(0) = -∞, log(pequeno_numero) = grande_negativo.
Solução: adicionar epsilon = 1e-15.

**Erro 3: Ignorar vanishing gradient**
Sigmoid tem derivada máxima 0.25. Em 10 camadas: 0.25^10 ≈ 10^-7!


### O que observar: Propriedades de Funções
- Continuidade vs descontinuidade afeta diferenciabilidade
- Monotonicidade determina se função é invertível
- Domínio restrito limita aplicabilidade em ML


### O que observar: Regressão Linear
- Reta ajustada minimiza soma de resíduos quadráticos
- R² próximo de 1 indica bom ajuste
- Outliers podem distorcer a reta significativamente


### O que observar: Softmax
- Exponenciais amplificam diferenças pequenas
- Softmax normaliza para distribuição de probabilidade
- Maior logit domina a probabilidade final


### O que observar: Cross-Entropy
- Erros confiantes têm penalidade exponencial
- Função côncava força calibração do modelo
- Fundamental para otimização em classificação


### O que observar: Funções de Ativação
- Sigmoid é suave mas sofre vanishing gradient
- ReLU é rápido mas pode matar neurônios
- Composição de ativações cria expressividade


### Por que em ML: Composição de Funções

1. **Redes neurais = composição:** f(w2 · σ(w1 · x + b1) + b2) é uma composição de lineares + não-lineares
2. **Sem ativações seria linear:** Se usasse só transformações lineares, rede = uma reta gigante
3. **Ativações quebram linearidade:** Sigmoid/ReLU adicionam curvatura, permitindo aprender padrões complexos
4. **Profundidade é poder:** Mais camadas = mais composições = funções mais complexas possíveis